In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats

from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve, classification_report, confusion_matrix

# Reading test, validation, and training data from files

df_test = pd.read_csv('/workspaces/churn-analysis/datasets/test.csv')
df_val = pd.read_csv('/workspaces/churn-analysis/datasets/val.csv')
df_train = pd.read_csv('/workspaces/churn-analysis/datasets/val.csv')

y_test = df_test['churn']
y_val = df_val['churn']
y_train = df_train['churn']

# First rows of the dataframes
print("Test Data:")
print(df_test.head(3))
      

df_test.drop(columns=['churn'], inplace=True)
df_val.drop(columns=['churn'], inplace=True)
df_train.drop(columns=['churn'], inplace=True) 

y_val = (y_val=='yes').astype(int)
y_test = (y_test=='yes').astype(int)



Test Data:
   gender  seniorcitizen partner dependents  tenure phoneservice  \
0  female              0      no         no      41          yes   
1  female              1      no         no      66          yes   
2  female              0      no         no      12          yes   

  multiplelines internetservice onlinesecurity onlinebackup deviceprotection  \
0            no             dsl            yes           no              yes   
1           yes     fiber_optic            yes           no               no   
2            no             dsl             no           no               no   

  techsupport streamingtv streamingmovies        contract paperlessbilling  \
0         yes         yes             yes        one_year              yes   
1          no         yes             yes        two_year              yes   
2          no          no              no  month-to-month              yes   

               paymentmethod  monthlycharges  totalcharges churn  
0  bank_transfe

In [4]:
def train(df_train, y_train, C=1.0):
    
    train_dicts = df_train.to_dict(orient='records')

    dv = DictVectorizer(sparse=False)
    X_train = dv.fit_transform(train_dicts)

    model = LogisticRegression(max_iter=5000, random_state=1, C=C)
    model.fit(X_train, y_train)
    
    return dv, model

def predict(df, dv, model):
    dicts = df.to_dict(orient='records')
    X = dv.transform(dicts)
    y_pred = model.predict_proba(X)[:, 1]
    return y_pred


dv, model = train(df_train, y_train)

val_dicts = df_val.to_dict(orient='records')
X_val = dv.transform(val_dicts)

test_dicts = df_test.to_dict(orient='records')
X_test = dv.transform(test_dicts)

dv.feature_names_

['contract=month-to-month',
 'contract=one_year',
 'contract=two_year',
 'dependents=no',
 'dependents=yes',
 'deviceprotection=no',
 'deviceprotection=no_internet_service',
 'deviceprotection=yes',
 'gender=female',
 'gender=male',
 'internetservice=dsl',
 'internetservice=fiber_optic',
 'internetservice=no',
 'monthlycharges',
 'multiplelines=no',
 'multiplelines=no_phone_service',
 'multiplelines=yes',
 'onlinebackup=no',
 'onlinebackup=no_internet_service',
 'onlinebackup=yes',
 'onlinesecurity=no',
 'onlinesecurity=no_internet_service',
 'onlinesecurity=yes',
 'paperlessbilling=no',
 'paperlessbilling=yes',
 'partner=no',
 'partner=yes',
 'paymentmethod=bank_transfer_(automatic)',
 'paymentmethod=credit_card_(automatic)',
 'paymentmethod=electronic_check',
 'paymentmethod=mailed_check',
 'phoneservice=no',
 'phoneservice=yes',
 'seniorcitizen',
 'streamingmovies=no',
 'streamingmovies=no_internet_service',
 'streamingmovies=yes',
 'streamingtv=no',
 'streamingtv=no_internet_servic

In [ ]:
for C in [0.001, 0.01, 0.1, 1, 10, 100, 1000]:

    dv, model = train(df_train, y_train, C=C)


    for k in np.arange(0, 1.0, 0.05):
        y_pred = (predict(df_val, dv, model) >= k).astype(int)
        print(f"C={C}: k={round(k, 2)}, accuracy={round((y_pred == y_val).mean(), 2)}")

dv, model = train(df_train, y_train, C=1)

C=0.001: k=0.0, accuracy=0.27
C=0.001: k=0.05, accuracy=0.41
C=0.001: k=0.1, accuracy=0.52
C=0.001: k=0.15, accuracy=0.62
C=0.001: k=0.2, accuracy=0.69
C=0.001: k=0.25, accuracy=0.72
C=0.001: k=0.3, accuracy=0.74
C=0.001: k=0.35, accuracy=0.76
C=0.001: k=0.4, accuracy=0.78
C=0.001: k=0.45, accuracy=0.79
C=0.001: k=0.5, accuracy=0.79
C=0.001: k=0.55, accuracy=0.8
C=0.001: k=0.6, accuracy=0.79
C=0.001: k=0.65, accuracy=0.77
C=0.001: k=0.7, accuracy=0.75
C=0.001: k=0.75, accuracy=0.74
C=0.001: k=0.8, accuracy=0.73
C=0.001: k=0.85, accuracy=0.73
C=0.001: k=0.9, accuracy=0.73
C=0.001: k=0.95, accuracy=0.73
C=0.01: k=0.0, accuracy=0.27
C=0.01: k=0.05, accuracy=0.45
C=0.01: k=0.1, accuracy=0.56
C=0.01: k=0.15, accuracy=0.64
C=0.01: k=0.2, accuracy=0.72
C=0.01: k=0.25, accuracy=0.74
C=0.01: k=0.3, accuracy=0.77
C=0.01: k=0.35, accuracy=0.79
C=0.01: k=0.4, accuracy=0.79
C=0.01: k=0.45, accuracy=0.8
C=0.01: k=0.5, accuracy=0.8
C=0.01: k=0.55, accuracy=0.81
C=0.01: k=0.6, accuracy=0.8
C=0.01: k=0

In [ ]:
# Choosing best threshold for validation set

for k in np.arange(0, 1.0, 0.05):
    y_pred = (predict(df_val, dv, model) >= k).astype(int)
    print(f"k={round(k, 2)}")
    print(pd.crosstab(y_val, y_pred))
    print(f"accuracy={round((y_pred == y_val).mean(), 2)}")

# Final model with C=1 and k=0.4 as it has the good accuracy and minor false negatives

k=0.0
col_0     1
churn      
0      1023
1       386
accuracy=0.27
k=0.05
col_0    0    1
churn          
0      371  652
1        8  378
accuracy=0.53
k=0.1
col_0    0    1
churn          
0      510  513
1       21  365
accuracy=0.62
k=0.15
col_0    0    1
churn          
0      617  406
1       36  350
accuracy=0.69
k=0.2
col_0    0    1
churn          
0      688  335
1       48  338
accuracy=0.73
k=0.25
col_0    0    1
churn          
0      751  272
1       66  320
accuracy=0.76
k=0.3
col_0    0    1
churn          
0      798  225
1       77  309
accuracy=0.79
k=0.35
col_0    0    1
churn          
0      834  189
1      100  286
accuracy=0.79
k=0.4
col_0    0    1
churn          
0      866  157
1      125  261
accuracy=0.8
k=0.45
col_0    0    1
churn          
0      893  130
1      145  241
accuracy=0.8
k=0.5
col_0    0    1
churn          
0      923  100
1      165  221
accuracy=0.81
k=0.55
col_0    0    1
churn          
0      947   76
1      185  201
accuracy=0.81
k=0.

In [10]:
# Saving the model and the DictVectorizer to files
import pickle

with open('/workspaces/churn-analysis/models/model_c1.bin', 'wb') as f_out:
    pickle.dump((dv, model), f_out)